In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [2]:
# Initialize Spark with Kafka and Postgres packages
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5a0c42f4-8c45-4f47-9875-8a195cf3920c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1362ms :: artifacts dl 30ms
	:: modules in u

In [3]:
print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [3]:
# 1. Load Static User Data (From CSV or Postgres)
# If using CSV: 
users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)


In [4]:
# 2. Read Streaming Data from Kafka
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
    ])

In [5]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection_sys") \
    .load()

In [6]:
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),tx_schema).alias("tx")).select("tx.*")
# Fraud Logic: Amount > 10,000
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [7]:
# 4. Enrich Stream with User Details (Stream-Static Join)
enriched_fraud = fraud_stream.join(users_df, "userId")

In [8]:
# 5. Format for output Kafka topic
output_stream = enriched_fraud \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

In [10]:
# output_stream.printSchema()
# output_stream.explain(True)

In [ ]:
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notis") \
    .option("checkpointLocation", "/workspace/checkpointsss") \
    .start()
query.awaitTermination()

26/06/22 03:35:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 03:35:00 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

In [ ]:
# print("Spark Version:", spark.version)
# print(query.exception())
# query.status
# query.lastProgress